- Activate environment
```
conda create -n experiment python=3.12
conda activate experiment
pip install -r requirements.txt
```

- Download test files
```
curl https://iotanalytics.unsw.edu.au/iottestbed/pcap/filelist.txt -o filelist.txt
head -20 filelist.txt | grep -Ev '(^#.*|^$)' filelist.txt | xargs -n 1 curl -O
```

- Untar file
```
for f in *.zip; do
    unzip "$f"
done
```

In [60]:
import pandas as pd
import os
from typing import Tuple, Set, Dict
from pathlib import Path
import numpy as np

- Load the data

In [61]:
CSV_DATASET_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv"
CSV_PROCESSED_DATA_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/processed-data/from-csv"
PCAP_DATASET_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap"
PCAP_PROCESSED_DATA_FOLDER = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/processed-data/from-pcap"
DEVICE_MAPPING_FILE = "../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/device-mapping/List_Of_Devices.txt"

- Assuming the last 9 devices are non-IoT with some caveats
    - These 9 devices include **Nest DropCam** which is not considered in the dataset because of its special traffic
    - **TP Link Router** is also will be taken off from the list.
- The final list will have 21 IoT devices and 7 non-IoT devices
    - The mapping will print 22 IoT devices because the Insteon Camera has 2 mapping for Wired and Wireless connection

In [62]:
def process_device_mapping(file_path: str) -> Tuple[Set[str], Set[str]]:
    """
    Process the device mapping file and return a set of IoT devices and a set of non-IoT devices.
    :param file_path: The path to the device mapping file.
    :return: A tuple containing a set of IoT devices and a set of non-IoT devices.
    """
    device_mapping = []
    with open(file_path, 'r') as f:
        for line in f:
            line_data = line.strip().split('\t')
            mapping = [data.strip() for data in line_data if data.strip()]
            if mapping and mapping[0].lower() != "list of devices":
                device_mapping.append((mapping[0], mapping[1]))

    deviceToBeRemoved = set(["14:cc:20:51:33:ea", "30:8c:fb:b6:ea:45"])
    all_devices = {mapping[1]: mapping[0] for mapping in device_mapping if mapping[1] not in deviceToBeRemoved}
    iot_devices = [mapping[0] for mapping in device_mapping[:-9] if mapping[1] not in deviceToBeRemoved]
    non_iot_devices = [mapping[0] for mapping in device_mapping[-9:] if mapping[1] not in deviceToBeRemoved]

    print(f"{len(iot_devices)} IoT devices: {iot_devices}")
    print(f"{len(non_iot_devices)} Non-IoT devices: {non_iot_devices}")
    print(f"{len(all_devices)} total devices: {all_devices}")

    return all_devices, iot_devices, non_iot_devices

all_devices, iot_devices, non_iot_devices = process_device_mapping(DEVICE_MAPPING_FILE)

22 IoT devices: ['Smart Things', 'Amazon Echo', 'Netatmo Welcome', 'TP-Link Day Night Cloud camera', 'Samsung SmartCam', 'Dropcam', 'Insteon Camera', 'Insteon Camera', 'Withings Smart Baby Monitor', 'Belkin Wemo switch', 'TP-Link Smart plug', 'iHome', 'Belkin wemo motion sensor', 'NEST Protect smoke alarm', 'Netatmo weather station', 'Withings Smart scale', 'Blipcare Blood Pressure meter', 'Withings Aura smart sleep sensor', 'Light Bulbs LiFX Smart Bulb', 'Triby Speaker', 'PIX-STAR Photo-frame', 'HP Printer']
7 Non-IoT devices: ['Samsung Galaxy Tab', 'Android Phone', 'Laptop', 'MacBook', 'Android Phone', 'IPhone', 'MacBook/Iphone']
29 total devices: {'d0:52:a8:00:67:5e': 'Smart Things', '44:65:0d:56:cc:d3': 'Amazon Echo', '70:ee:50:18:34:43': 'Netatmo Welcome', 'f4:f2:6d:93:51:f1': 'TP-Link Day Night Cloud camera', '00:16:6c:ab:6b:88': 'Samsung SmartCam', '30:8c:fb:2f:e4:b2': 'Dropcam', '00:62:6e:51:27:2e': 'Insteon Camera', 'e8:ab:fa:19:de:4f': 'Insteon Camera', '00:24:e4:11:18:a8': '

In [63]:
csv_files = [f"{CSV_DATASET_FOLDER}/{f.name}" for f in sorted(Path(CSV_DATASET_FOLDER).iterdir()) if f.is_file() and f.suffix == ".csv"]
for file in csv_files:
    print(file)

../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-23.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-24.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-25.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-26.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-27.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-28.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-29.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-30.csv
../../dataset/identify-iot-devices-and-events-based-on-p

- The `TIME` columns in already in Unix time seconds format
- According to the paper, we will be grouping traffic with the same second window and originating from the same MAC source address. Based on the IoT and non-IoT devices list above, we'll label this traffic if it is IoT or non-IoT

In [64]:
def preprocess_csv(file_path: str, all_devices: Dict[str, str]) -> pd.DataFrame:
    """
    Preprocess the CSV file and return a DataFrame with the relevant features.
    :param file_path: The path to the CSV file.
    :param all_devices: A dictionary mapping device MAC addresses to names.
    :return: A preprocessed DataFrame with the relevant features.
    """
    df = pd.read_csv(file_path)
    
    # First map eth.src to device name
    df["device_name"] = df["eth.src"].map(all_devices).fillna("Unknown")
    filtered_unknown_df = df[df["device_name"] != "Unknown"]
    features_df = (filtered_unknown_df.groupby(['device_name', 'TIME'])["Size"].agg( mean="mean", std=lambda x: x.std(ddof=0), total_bytes="sum", packet_count="size",).reset_index())
    print(f"Preprocessed {file_path}: {len(features_df)} rows.")

    return features_df

In [65]:
features_extracted_dfs = []
for file in csv_files:
    processed_df = preprocess_csv(file, all_devices)
    if processed_df is not None and not processed_df.empty:
        features_extracted_dfs.append(processed_df)

Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-23.csv: 154260 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-24.csv: 146743 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-25.csv: 136969 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-26.csv: 140156 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-27.csv: 140052 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-28.csv: 179669 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/csv/16-09-29.csv

In [66]:
features_df = pd.concat(features_extracted_dfs, ignore_index=True)

In [ ]:
features_df.shape

(3112196, 5)

In [67]:
features_df[(features_df["packet_count"] == 1)].shape

(2205664, 6)

- 3 millions records doesn't look like what the paper computes. Now, we'll start looking into the raw pcap file and use a different way of calculating the window.
- Untar the pcap files
```bash
for f in *.tar.gz; do
    tar -xzf "$f"
done
```
- Parse the pcap files and export them to csv file
```bash
for f in *.pcap; do
    echo -e "Processing file ${f}...."
    tshark -r ${f} \
    -T fields \
    -E header=y \
    -E separator=, \
    -E quote=d \
    -e frame.number \
    -e frame.time_epoch \
    -e frame.len \
    -e eth.src \
    -e eth.dst \
    -e ip.src \
    -e ip.dst \
    -e ip.proto \
    -e tcp.srcport \
    -e tcp.dstport \
    -e udp.srcport \
    -e udp.dstport \
    > ${f}.csv
done
```

In [42]:
pcap_csv_files = [f"{PCAP_DATASET_FOLDER}/{f.name}" for f in sorted(Path(PCAP_DATASET_FOLDER).iterdir()) if f.is_file() and f.suffix == ".csv"]
for file in pcap_csv_files:
    print(file)

../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-23.pcap.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-24.pcap.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-25.pcap.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-26.pcap.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-27.pcap.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-28.pcap.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-29.pcap.csv
../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-30.pcap.csv
../../da

- Now instead of fixed time windows, we are processing the data by packet relative time window

In [43]:
def assign_packet_triggered_windows(
    timestamps: pd.Series,
) -> pd.Series:
    values = timestamps.to_numpy(dtype=float)

    window_ids = np.zeros(len(values), dtype=np.int64)

    if len(values) == 0:
        return pd.Series(window_ids, index=timestamps.index)

    current_window = 0
    window_start = values[0]

    for i, timestamp in enumerate(values):
        if timestamp >= window_start + 1.0:
            current_window += 1
            window_start = timestamp

        window_ids[i] = current_window

    return pd.Series(window_ids, index=timestamps.index)

def preprocess_pcap_csv(file_path: str, all_devices: Dict[str, str]) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    df["device_name"] = df["eth.src"].map(all_devices).fillna("Unknown")
    filtered_unknown_df = df[df["device_name"] != "Unknown"].dropna(subset=["device_name", "frame.time_epoch", "frame.len"])
    sorted_filtered_df = filtered_unknown_df.sort_values(["device_name", "frame.time_epoch"])
    sorted_filtered_df["window"] = (
        sorted_filtered_df.groupby("device_name", group_keys=False)["frame.time_epoch"]
        .apply(assign_packet_triggered_windows)
    )
    features_df = (
        sorted_filtered_df.groupby(["device_name", "window"])["frame.len"]
        .agg(
            mean="mean",
            std=lambda x: x.std(ddof=0),
            total_bytes="sum",
            packet_count="size",
        )
        .reset_index()
    )
    print(f"Preprocessed {file_path}: {len(features_df)} rows.")

    return features_df

In [44]:
features_extracted_pcap_dfs = []
for file in pcap_csv_files:
    processed_df = preprocess_pcap_csv(file, all_devices)
    if processed_df is not None and not processed_df.empty:
        features_extracted_pcap_dfs.append(processed_df)

Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-23.pcap.csv: 180890 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-24.pcap.csv: 170971 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-25.pcap.csv: 161759 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-26.pcap.csv: 164524 rows.


/var/folders/rv/241jb89d0rn0z37gwxcvxhxc0000gn/T/ipykernel_41417/1895392433.py:24: DtypeWarning: Columns (0: ip.proto) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-27.pcap.csv: 165184 rows.


/var/folders/rv/241jb89d0rn0z37gwxcvxhxc0000gn/T/ipykernel_41417/1895392433.py:24: DtypeWarning: Columns (0: ip.proto) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-28.pcap.csv: 201844 rows.
Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-09-29.pcap.csv: 199265 rows.


/var/folders/rv/241jb89d0rn0z37gwxcvxhxc0000gn/T/ipykernel_41417/1895392433.py:24: DtypeWarning: Columns (0: ip.proto) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


... (15 outputs left out)

/var/folders/rv/241jb89d0rn0z37gwxcvxhxc0000gn/T/ipykernel_41417/1895392433.py:24: DtypeWarning: Columns (0: ip.proto) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-10-11.pcap.csv: 188652 rows.


/var/folders/rv/241jb89d0rn0z37gwxcvxhxc0000gn/T/ipykernel_41417/1895392433.py:24: DtypeWarning: Columns (0: ip.proto) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Preprocessed ../../dataset/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/raw-data/pcap/16-10-12.pcap.csv: 105100 rows.


In [45]:
features_extracted_pcap_df = pd.concat(features_extracted_pcap_dfs, ignore_index=True)

In [49]:
features_extracted_pcap_df[(features_extracted_pcap_df["packet_count"] == 1)].shape

(2329982, 6)

In [50]:
features_extracted_pcap_df.shape

(3607698, 6)